# GPT 시리즈: 언어모델의 스케일링
## 실습 코드 1: GPT-2로 Zero-shot / Few-shot 프롬프팅

---

### 🎯 이 노트북에서 배우는 것

| 순서 | 주제 | 핵심 질문 |
|:---:|---|---|
| 1 | Logit과 확률 분포 | 모델이 출력하는 숫자가 왜 확률이 아닌가? |
| 2 | Temperature | 숫자 하나로 모델이 얼마나 '대담해'지는가? |
| 3 | Top-k / Top-p 샘플링 | 어떻게 품질과 다양성을 동시에 얻는가? |
| 4 | Zero-shot | 예시 없이 지시만으로 작동하는가? |
| 5 | Few-shot | 예시 몇 개가 얼마나 도움이 되는가? |

---

### 📌 읽는 방법

- **셀을 위에서부터 순서대로** 실행하세요. 앞 셀의 결과가 뒤 셀에서 사용됩니다.
- 주석(`#`)이 달린 줄은 해당 줄이 **왜** 필요한지 설명합니다. 코드가 낯설면 주석부터 읽으세요.
- `# ▶ 실험` 표시가 있는 곳은 숫자나 텍스트를 바꿔 결과 변화를 직접 확인해 보세요.

---

### ⚙️ 실행 환경

- Google Colab 또는 로컬 Jupyter 환경을 권장합니다.
- GPU가 있으면 빠르지만, CPU만으로도 실습 가능합니다.
- 필요 패키지: `transformers`, `torch`, `matplotlib`

---
## Cell 1 | 패키지 설치 & 임포트

가장 먼저 실습에 필요한 라이브러리를 설치하고 불러옵니다.

- **transformers**: Hugging Face에서 만든 라이브러리. GPT-2 같은 사전학습 모델을 단 두 줄로 불러올 수 있습니다.
- **torch (PyTorch)**: 모델 내부의 텐서(다차원 배열) 연산을 담당합니다.
- **matplotlib**: 확률 분포를 막대그래프로 시각화하는 데 사용합니다.

In [ ]:
# Colab 환경에서는 아래 설치 명령을 먼저 실행하세요.
# 이미 설치되어 있다면 이 셀은 건너뛰어도 됩니다.
# !pip install transformers torch matplotlib --quiet

# ── 임포트 ───────────────────────────────────────────────────
from transformers import GPT2LMHeadModel, GPT2Tokenizer
# GPT2LMHeadModel : GPT-2 언어 모델 본체 (파라미터가 저장된 거대한 행렬 묶음)
# GPT2Tokenizer   : 텍스트 ↔ 정수 ID 변환 담당 (어휘 사전 포함)

import torch                          # 텐서 연산 라이브러리
import torch.nn.functional as F       # softmax 등 수학 함수 모음
import matplotlib.pyplot as plt       # 그래프 시각화
import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False  # 마이너스 부호 깨짐 방지

print("✅ 임포트 완료")

---
## Cell 2 | 모델 & 토크나이저 불러오기

GPT-2는 여러 크기의 버전이 있습니다.

| 모델 이름 | 파라미터 수 | 특징 |
|---|---|---|
| `gpt2` | 117M | 가장 작고 빠름. 실습 시작용으로 적합 |
| `gpt2-medium` | 355M | 성능과 속도의 균형 |
| `gpt2-large` | 774M | 품질이 높지만 느림 |
| `gpt2-xl` | 1.5B | 가장 크고 느림. GPU 권장 |

> **처음에는 `gpt2` 또는 `gpt2-medium`으로 시작하세요.** 너무 큰 모델은 CPU에서 수십 초씩 걸립니다.

In [ ]:
# ── 모델 선택 ────────────────────────────────────────────────
# ▶ 실험: 아래 이름을 'gpt2', 'gpt2-large' 등으로 바꿔보세요.
MODEL_NAME = "gpt2-medium"  # 355M 파라미터

print(f"모델 '{MODEL_NAME}' 다운로드 중... (최초 실행 시 수십 초 소요)")

# ── 토크나이저 로드 ──────────────────────────────────────────
# 토크나이저는 텍스트를 모델이 이해하는 정수 토큰 ID 목록으로 변환합니다.
# 예) "Hello world" → [15496, 995]
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)

# ── 모델 로드 ────────────────────────────────────────────────
# 사전학습된 가중치(파라미터)를 포함한 GPT-2 모델 전체를 불러옵니다.
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)

# ── 추론 모드 설정 ───────────────────────────────────────────
# model.eval()을 호출하면 Dropout, BatchNorm 같은 학습용 레이어가 비활성화됩니다.
# 우리는 학습이 아닌 '추론(inference)'만 할 것이므로 반드시 호출해야 합니다.
model.eval()

# ── GPU 사용 여부 확인 ───────────────────────────────────────
# GPU가 있으면 연산이 수십 배 빠릅니다. 없으면 CPU로 자동 전환됩니다.
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)  # 모델을 선택한 장치(GPU/CPU)로 이동

total_params = sum(p.numel() for p in model.parameters())
print(f"\n✅ 로드 완료!")
print(f"   사용 장치    : {device.upper()}")
print(f"   모델 파라미터: {total_params/1e6:.0f}M")

---
## Cell 3 | 토크나이저 동작 이해하기

모델은 텍스트를 직접 읽지 못합니다. 먼저 **토크나이저(Tokenizer)** 가 텍스트를 정수 ID 목록으로 변환해야 합니다.

```
"Hello, world!"  →  토크나이저  →  [15496, 11, 995, 0]
     텍스트                            토큰 ID 목록
```

토큰은 단어 전체일 수도 있고, 단어의 일부 조각일 수도 있습니다.  
GPT-2는 약 **50,257개**의 토큰 어휘를 사용합니다.

In [ ]:
# ── 토크나이저 동작 직접 확인 ────────────────────────────────
sample_text = "The quick brown fox"

# 텍스트 → 토큰 ID
token_ids = tokenizer.encode(sample_text)
print(f"입력 텍스트  : '{sample_text}'")
print(f"토큰 ID 목록 : {token_ids}")
print(f"토큰 개수    : {len(token_ids)}개")

# 각 ID가 어떤 텍스트 조각(토큰)에 해당하는지 확인
print("\n각 토큰 상세:")
print(f"  {'ID':>6}  →  텍스트")
print(f"  {'──':>6}     ────────")
for tid in token_ids:
    # tokenizer.decode()는 ID → 텍스트 역변환
    text_piece = tokenizer.decode([tid])
    print(f"  {tid:>6}  →  '{text_piece}'")

# 토큰 ID → 다시 텍스트로 복원
restored = tokenizer.decode(token_ids)
print(f"\n복원된 텍스트: '{restored}'")
print(f"\nGPT-2 어휘 크기: {tokenizer.vocab_size:,}개 토큰")

---
## Cell 4 | Logit이란 무엇인가?

모델의 핵심 출력값인 **logit**을 이해하는 것이 이 실습의 첫 번째 목표입니다.

```
입력 텍스트 → [토크나이저] → 토큰 ID 배열 → [GPT-2 모델] → logit 배열 → [Softmax] → 확률 분포
```

**Logit**: 모델이 각 토큰을 '다음에 올 단어'로 얼마나 선호하는지를 나타내는 **정규화되지 않은 점수**입니다.
- logit의 범위: 음수 ∞ ~ 양수 ∞ (확률이 아닙니다)
- logit 값이 클수록 해당 토큰이 선택될 가능성이 높습니다
- **Softmax** 함수를 통과하면 비로소 0~1 사이의 확률로 변환됩니다

In [ ]:
# ── Logit 직접 추출 & 확률 변환 관찰 ──────────────────────────

# 프롬프트 준비
prompt = "The weather today is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
# return_tensors="pt" : 파이썬 리스트가 아닌 PyTorch 텐서로 반환하라는 의미

# 모델 순방향 전파 (gradient 계산 불필요 → no_grad로 메모리 절약)
with torch.no_grad():
    outputs = model(**inputs)
    # outputs.logits shape: (batch_size=1, seq_len, vocab_size=50257)
    # 우리가 관심 있는 것은 '마지막 토큰 다음'에 올 단어의 logit
    last_logits = outputs.logits[0, -1, :]  # shape: (50257,)

print(f"프롬프트: '{prompt}'")
print(f"\n[Logit 정보]")
print(f"  logit 벡터 길이 : {last_logits.shape[0]:,}  ← 어휘 크기와 같음")
print(f"  logit 최댓값    : {last_logits.max().item():.3f}")
print(f"  logit 최솟값    : {last_logits.min().item():.3f}")
print(f"  logit 합계      : {last_logits.sum().item():.3f}  ← 1이 아님! (아직 확률 아님)")

# ── Softmax로 확률 변환 ──────────────────────────────────────
# Softmax 공식:  P(i) = exp(logit_i) / Σ exp(logit_j)
# 모든 값을 양수로 만들고 전체 합이 1이 되도록 정규화합니다.
probs = F.softmax(last_logits, dim=-1)

print(f"\n[Softmax 변환 후 확률]")
print(f"  확률 최댓값 : {probs.max().item():.4f}")
print(f"  확률 최솟값 : {probs.min().item():.8f}")
print(f"  확률 합계   : {probs.sum().item():.6f}  ← 1에 수렴 ✅")

# ── 상위 10개 후보 토큰 출력 ──────────────────────────────────
top10_probs, top10_ids = torch.topk(probs, 10)

print(f"\n[상위 10개 후보 토큰] (프롬프트 다음에 올 단어)")
print(f"  {'순위':>4}  {'토큰':>12}  {'확률':>8}  {'logit':>8}")
print(f"  {'──':>4}  {'────':>12}  {'────':>8}  {'─────':>8}")
for rank, (prob, tid) in enumerate(zip(top10_probs, top10_ids), 1):
    token_text = tokenizer.decode([tid.item()])
    logit_val  = last_logits[tid].item()
    print(f"  {rank:>4}  {repr(token_text):>12}  {prob.item():>8.4f}  {logit_val:>8.3f}")

---
## Cell 5 | Temperature — '창의성'을 조절하는 단 하나의 숫자

**Temperature**는 Softmax를 적용하기 **전에** logit을 나눠주는 값입니다.

$$P(i) = \frac{\exp(logit_i \;/\; T)}{\sum_j \exp(logit_j \;/\; T)}$$

| Temperature | 효과 | 비유 |
|:---:|---|---|
| `T < 1.0` | 확률 분포가 뾰족해짐 → 상위 토큰에 쏠림 → **결정적**, 반복적 | 신중한 사람 |
| `T = 1.0` | 원래 logit 그대로 | 기본값 |
| `T > 1.0` | 확률 분포가 평탄해짐 → 낮은 순위 토큰도 선택됨 → **창의적**, 예측 불가 | 즉흥적인 사람 |

In [ ]:
# ── Temperature 효과를 숫자와 그래프로 확인 ───────────────────

# 예시용 작은 logit 벡터 (5개 토큰만)
# 실제로는 50,257개지만 시각화를 위해 5개로 줄임
demo_logits = torch.tensor([3.2, 1.5, 0.8, 2.1, -0.5])
demo_tokens = ["sunny", "cloudy", "rainy", "warm", "cold"]

# ▶ 실험: 아래 temperature 값을 변경하면서 확률 분포가 어떻게 달라지는지 보세요
temperatures = [0.3, 0.7, 1.0, 1.5, 2.5]

print("Temperature에 따른 확률 분포 변화")
print(f"\n  원본 logit: {demo_logits.tolist()}")
print(f"  토큰 이름  : {demo_tokens}")
print()

fig, axes = plt.subplots(1, len(temperatures), figsize=(15, 4))
fig.suptitle("Temperature에 따른 확률 분포 변화", fontsize=13, y=1.02)

for ax, T in zip(axes, temperatures):
    # logit을 T로 나눈 뒤 Softmax 적용
    scaled_logits = demo_logits / T
    probs_t = F.softmax(scaled_logits, dim=-1).numpy()

    # 텍스트 출력
    probs_str = ", ".join(f"{p:.3f}" for p in probs_t)
    max_prob = probs_t.max()
    print(f"  T={T:>4}: [{probs_str}]  max={max_prob:.3f}")

    # 막대그래프
    colors = ["steelblue"] * 5
    colors[int(probs_t.argmax())] = "tomato"  # 가장 높은 확률은 빨간색
    bars = ax.bar(demo_tokens, probs_t, color=colors, edgecolor="white")
    ax.set_title(f"T = {T}", fontsize=12, fontweight="bold")
    ax.set_ylim(0, 1.0)
    ax.set_ylabel("확률")
    ax.tick_params(axis='x', rotation=30)

    # 각 막대 위에 확률 값 표시
    for bar, prob in zip(bars, probs_t):
        if prob > 0.03:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f"{prob:.2f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

print()
print("관찰 포인트:")
print("  • T가 작을수록 → 막대 하나(빨강)에 확률이 집중됩니다 (결정적)")
print("  • T가 클수록   → 막대들의 높이가 비슷해집니다 (무작위/창의적)")
print("  • T=1.0이 logit 그대로의 기본 상태입니다")

---
## Cell 6 | Top-k & Top-p — 허용할 후보를 걸러내는 두 가지 방법

Softmax 확률 분포에서 단순히 가장 높은 토큰만 뽑으면 반복적이고 지루해집니다.  
그렇다고 전체 50,257개 중 무작위로 뽑으면 이상한 단어가 나올 수 있습니다.

이 둘 사이의 균형을 잡는 두 가지 방법이 **Top-k**와 **Top-p**입니다.

### Top-k 샘플링
확률 상위 `k`개 토큰만 후보로 남기고, 나머지는 `-∞`로 마스킹합니다.

```
전체 50,257개 토큰  →  상위 k개만 남김  →  그 k개 중에서 확률적으로 선택
```

### Top-p (Nucleus) 샘플링
누적 확률이 `p`가 되는 시점까지만 토큰을 후보로 유지합니다.  
상황에 따라 후보 수가 동적으로 변하기 때문에 더 유연합니다.

```
확률이 높은 토큰부터 더해서 → 합계가 p(예: 0.9)에 도달할 때까지 선택 → 나머지 제거
```

In [ ]:
# ── Top-k vs Top-p 동작 원리를 단계별로 구현 ─────────────────

# 실제 GPT-2 logit을 사용 (앞 Cell 4에서 구한 last_logits)
raw_logits = last_logits.clone().cpu()  # 원본 보존을 위해 복사

# Softmax → 확률 분포
base_probs = F.softmax(raw_logits, dim=-1)


# ┌─────────────────────────────────────────────────────────┐
# │  Helper: 상위 N개 후보를 출력하는 함수                  │
# └─────────────────────────────────────────────────────────┘
def show_top_tokens(probs_tensor, label, n=8):
    top_probs, top_ids = torch.topk(probs_tensor, n)
    print(f"\n  [{label}] 상위 {n}개 후보")
    print(f"  {'토큰':>12}  {'확률':>8}")
    print(f"  {'────':>12}  {'────':>8}")
    # 합계가 1이 아닌 경우를 대비해 재정규화
    total = top_probs.sum().item()
    for prob, tid in zip(top_probs, top_ids):
        token_text = tokenizer.decode([tid.item()])
        print(f"  {repr(token_text):>12}  {prob.item():>8.4f}")
    remaining = (probs_tensor > 0).sum().item()
    print(f"  → 후보 토큰 총 {remaining:,}개 / 어휘 {len(probs_tensor):,}개")


# ── ① 필터링 없음 (기준선) ───────────────────────────────────
show_top_tokens(base_probs, "필터 없음 (기준)", n=8)


# ── ② Top-k 적용 ────────────────────────────────────────────
# ▶ 실험: k 값을 5, 20, 100 등으로 바꿔보세요
k = 10

# 상위 k개 값과 인덱스 추출
topk_vals, topk_idx = torch.topk(raw_logits, k)

# 모든 logit을 -inf로 초기화 (= 확률 0)
masked_logits_topk = torch.full_like(raw_logits, float('-inf'))

# 상위 k개 위치에만 원래 logit 값을 복원
masked_logits_topk[topk_idx] = raw_logits[topk_idx]

# 마스킹된 logit에 Softmax 적용 → k개 중에서만 확률 분배
probs_topk = F.softmax(masked_logits_topk, dim=-1)

show_top_tokens(probs_topk, f"Top-k (k={k})", n=k)


# ── ③ Top-p (Nucleus) 적용 ───────────────────────────────────
# ▶ 실험: p 값을 0.5, 0.95 등으로 바꿔보세요
p = 0.9

# 확률이 높은 순서로 정렬
sorted_probs, sorted_idx = torch.sort(base_probs, descending=True)

# 누적 확률 계산
cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

# 누적 확률이 p를 초과하는 순간의 토큰부터 제거
# (shift by 1: 현재 토큰 포함 시점에서 판단하므로 -sorted_probs로 조정)
remove_mask = (cumulative_probs - sorted_probs) > p

# -inf 마스킹 적용
masked_logits_topp = torch.full_like(raw_logits, float('-inf'))
kept_idx = sorted_idx[~remove_mask]  # 살아남은 토큰 인덱스
masked_logits_topp[kept_idx] = raw_logits[kept_idx]

probs_topp = F.softmax(masked_logits_topp, dim=-1)

kept_count = (~remove_mask).sum().item()
print(f"\n  [Top-p (p={p})] 상위 {kept_count}개 후보")
show_top_tokens(probs_topp, f"Top-p (p={p})", n=min(kept_count, 10))

print()
print("핵심 차이 정리:")
print(f"  Top-k : 항상 정확히 {k}개만 후보 (고정)")
print(f"  Top-p : 상황에 따라 후보 수가 달라짐 (이번에는 {kept_count}개)")

---
## Cell 7 | 완성된 `generate()` 함수 — 모든 개념을 하나로

앞에서 배운 temperature, top-k, top-p를 모두 파라미터로 받는 generate 함수를 만듭니다.  
내부 로직에 상세 주석을 달아두었으니, 동작 원리를 단계별로 추적해 보세요.

In [ ]:
def generate(
    prompt: str,
    max_new_tokens: int = 50,
    temperature: float = 0.8,
    top_k: int = 50,
    top_p: float = 0.9,
    do_sample: bool = True,
) -> str:
    """
    텍스트를 받아 GPT-2로 이어지는 내용을 생성합니다.

    Parameters
    ----------
    prompt        : 모델에게 줄 시작 텍스트
    max_new_tokens: 새로 생성할 최대 토큰 수 (프롬프트 길이 포함 X)
    temperature   : 낮을수록 결정적, 높을수록 창의적 (권장 범위: 0.3 ~ 1.5)
    top_k         : 후보 토큰 최대 개수 (0이면 top-k 비활성화)
    top_p         : 누적 확률 임계값 (1.0이면 top-p 비활성화)
    do_sample     : True면 확률적 샘플링, False면 항상 최고 확률 토큰 선택 (greedy)
    """
    # ── Step 1: 텍스트 → 토큰 ID 텐서 ───────────────────────
    inputs = tokenizer(
        prompt,
        return_tensors="pt",  # PyTorch 텐서 형식
    ).to(device)

    prompt_length = inputs["input_ids"].shape[1]  # 프롬프트 토큰 수

    # ── Step 2: 모델로 텍스트 생성 ──────────────────────────
    # gradient 계산 불필요 → no_grad로 메모리·속도 최적화
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,  # 새로 만들 토큰 수

            # ─ 샘플링 방식 설정 ─
            do_sample=do_sample,            # True: 샘플링  /  False: greedy
            temperature=temperature,        # logit 스케일 조정 (낮으면 뾰족, 높으면 평탄)
            top_k=top_k,                    # 상위 k개 토큰만 후보로 유지
            top_p=top_p,                    # 누적 확률 p 이내의 토큰만 후보로 유지

            # ─ EOS / PAD 처리 ─
            # GPT-2는 PAD 토큰이 없어서 EOS 토큰으로 대체
            pad_token_id=tokenizer.eos_token_id,
        )
        # output_ids shape: (1, prompt_len + new_tokens)

    # ── Step 3: 새로 생성된 토큰만 잘라내어 텍스트로 변환 ────
    new_token_ids = output_ids[0, prompt_length:]  # 프롬프트 이후 부분만
    generated_text = tokenizer.decode(
        new_token_ids,
        skip_special_tokens=True,  # EOS 같은 특수 토큰은 제외
    )

    return generated_text.strip()


# ── 동작 확인 ────────────────────────────────────────────────
test_result = generate("Once upon a time", max_new_tokens=20)
print("✅ generate() 함수 정상 동작 확인")
print(f"   입력  : 'Once upon a time'")
print(f"   출력  : '{test_result}'")

---
## Cell 8 | Zero-shot 프롬프팅 — 예시 없이 지시만으로

**Zero-shot 프롬프팅**: 예시(example)를 전혀 제공하지 않고, 작업 지시만 텍스트로 주는 방식입니다.

```
[작업 설명]
[입력]
[출력을 시작하는 단서(cue)]
```

> **GPT-2는 Zero-shot에 약합니다.** 2019년에 등장한 모델로, 지시 따르기(instruction-following)를 특별히 학습하지 않았기 때문입니다.  
> 같은 프롬프트를 GPT-3/4에 주면 훨씬 잘 따릅니다. 차이를 비교하는 것이 이 실습의 목적 중 하나입니다.

In [ ]:
# ── Zero-shot 프롬프트 모음 ───────────────────────────────────

# 각 프롬프트 구조를 주석으로 설명합니다.

zero_shot_prompts = {
    "번역 (영→불)": (
        # 구조: '작업 설명' + '입력' + '출력 단서'
        "Translate English to French:\n"
        "Cheese =>"  # '=>' 가 출력을 유도하는 단서(cue)
    ),
    "감정 분석": (
        # 구조: '작업 설명' + '입력' + '출력 단서'
        "Analyze the sentiment of this review. Answer with only 'Positive' or 'Negative'.\n"
        "Review: This product is absolutely awful.\n"
        "Sentiment:"
    ),
    "요약": (
        # 구조: '긴 입력 텍스트' + '요약 단서'
        "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. "
        "It is named after the engineer Gustave Eiffel, whose company designed and built the tower. "
        "Constructed from 1887 to 1889, it was initially criticized by some of France's leading artists "
        "and intellectuals for its design.\n"
        "TL;DR:"  # 'TL;DR'은 '요약해줘'를 의미하는 인터넷 관용 표현 → 모델이 요약을 이어쓰도록 유도
    ),
}

print("═" * 60)
print("           Zero-shot 프롬프팅 결과".center(60))
print("═" * 60)

for task_name, prompt in zero_shot_prompts.items():
    print(f"\n▶ 작업: {task_name}")
    print(f"  프롬프트:\n    {repr(prompt[:80])}{'...' if len(prompt) > 80 else ''}")
    result = generate(prompt, max_new_tokens=20, temperature=0.5)  # 낮은 T로 더 결정적으로
    print(f"  GPT-2 출력: '{result[:80]}'")

print()
print("─" * 60)
print("💡 GPT-2 한계 관찰:")
print("   GPT-2는 '지시 따르기'를 위해 학습된 모델이 아닙니다.")
print("   프롬프트를 지시로 인식하기보다 그냥 '이어지는 텍스트'로 처리합니다.")
print("   GPT-3 이후 RLHF(인간 피드백 강화학습)로 지시 따르기가 크게 개선되었습니다.")

---
## Cell 9 | Few-shot 프롬프팅 — 예시로 패턴을 가르치기

**Few-shot 프롬프팅**: 작업 예시를 **몇 개(few)** 프롬프트 안에 포함시켜 모델에게 "이런 형식으로 답해라"라는 패턴을 보여주는 방식입니다.

```
[예시 1: 입력] → [예시 1: 정답]
[예시 2: 입력] → [예시 2: 정답]
[예시 3: 입력] → [예시 3: 정답]
[실제 질문   ] → ???     ← 모델이 패턴을 따라 여기를 채웁니다
```

> **GPT-2에서도 Few-shot은 Zero-shot보다 잘 작동합니다.**  
> 모델이 파라미터를 업데이트(학습)하지 않고, 오직 맥락(context)만으로 패턴을 파악하기 때문에  
> 이를 **In-Context Learning(ICL)** 이라고도 부릅니다.

In [ ]:
# ── Few-shot 프롬프트 모음 ────────────────────────────────────

few_shot_prompts = {
    "번역 (영→불) — One-shot": (
        # 예시 1개만 제공 (One-shot)
        "Translate English to French:\n"
        "Cat => Chat\n"          # 예시: Cat → Chat
        "Dog =>"                  # 모델이 이 패턴을 보고 Chien을 출력해야 함
    ),
    "번역 (영→불) — Few-shot": (
        # 예시 3개 제공 (Few-shot)
        "Translate English to French:\n"
        "Cat => Chat\n"
        "Dog => Chien\n"
        "House => Maison\n"
        "Book =>"                 # 모델이 Livre를 출력해야 함
    ),
    "감정 분석 — Few-shot": (
        # 긍정/부정 레이블 예시를 주고, 마지막 리뷰의 감정을 분류하도록 유도
        "Review: This movie was terrible and boring.\n"
        "Sentiment: Negative\n"
        "\n"
        "Review: I loved this film, it was amazing!\n"
        "Sentiment: Positive\n"
        "\n"
        "Review: The food was okay, nothing special.\n"
        "Sentiment:"              # Neutral 또는 Negative가 적절한 답
    ),
    "수식 계산 — Few-shot": (
        # 수식 → 결과 패턴 예시
        "Q: 2 + 2 = A: 4\n"
        "Q: 5 + 3 = A: 8\n"
        "Q: 10 - 4 = A: 6\n"
        "Q: 7 + 5 = A:"           # 모델이 12를 출력해야 함
    ),
}

print("═" * 60)
print("          Few-shot 프롬프팅 결과".center(60))
print("═" * 60)

for task_name, prompt in few_shot_prompts.items():
    print(f"\n▶ 작업: {task_name}")

    # 프롬프트 내 예시 부분을 보기 좋게 출력
    lines = prompt.strip().split("\n")
    for line in lines:
        print(f"    {line}")

    result = generate(prompt, max_new_tokens=12, temperature=0.3)  # 낮은 T → 패턴을 잘 따름
    print(f"  → GPT-2 출력: '{result[:60]}'")

print()
print("─" * 60)
print("💡 Zero-shot vs Few-shot 차이:")
print("   Few-shot에서는 형식(Format) 예시를 몇 개만 줘도")
print("   모델이 그 패턴을 이어받아 올바른 위치에 답을 생성합니다.")

---
## Cell 10 | Zero-shot vs One-shot vs Few-shot 직접 비교

같은 작업(번역)을 예시 수에 따라 비교해 보겠습니다.  
예시 수가 늘어날수록 출력 품질이 어떻게 달라지는지 직접 관찰하세요.

In [ ]:
# ── 같은 작업을 Zero / One / Few-shot으로 비교 ───────────────

# 최종 질문: 'Water'를 프랑스어로 번역하면?
# 정답: Eau

prompts_by_shot = {
    "Zero-shot\n(예시 0개)": (
        "Translate English to French:\n"
        "Water =>"
    ),
    "One-shot\n(예시 1개)": (
        "Translate English to French:\n"
        "Cat => Chat\n"
        "Water =>"
    ),
    "Few-shot\n(예시 3개)": (
        "Translate English to French:\n"
        "Cat => Chat\n"
        "Dog => Chien\n"
        "House => Maison\n"
        "Water =>"
    ),
    "Few-shot\n(예시 5개)": (
        "Translate English to French:\n"
        "Cat => Chat\n"
        "Dog => Chien\n"
        "House => Maison\n"
        "Book => Livre\n"
        "Sun => Soleil\n"
        "Water =>"
    ),
}

print("═" * 65)
print(f" 작업: 'Water'를 프랑스어로 번역  (정답: Eau)")
print("═" * 65)

# 동일한 조건(temperature=0.3)으로 여러 번 실행해 안정성도 관찰
N_RUNS = 3  # ▶ 실험: 늘리면 출력의 다양성을 더 많이 볼 수 있습니다

for shot_label, prompt in prompts_by_shot.items():
    shot_label_clean = shot_label.replace("\n", " ")
    print(f"\n  {shot_label_clean}")
    results = []
    for i in range(N_RUNS):
        res = generate(prompt, max_new_tokens=8, temperature=0.3)
        # 첫 단어만 추출 (번역 결과는 보통 한 단어)
        first_word = res.split()[0] if res.split() else "(빈 출력)"
        results.append(first_word)
    print(f"    출력 {N_RUNS}회: {results}")
    correct = sum(1 for r in results if r.lower().startswith("eau"))
    print(f"    정답률: {correct}/{N_RUNS}")

print()
print("─" * 65)
print("💡 관찰: 예시가 많을수록 정답(Eau)이 나올 확률이 높아지나요?")

---
## Cell 11 | Temperature가 생성 결과에 미치는 영향 — 실전 비교

앞에서 이론으로 배운 temperature를 실제 문장 생성에서 확인합니다.

- **낮은 T (0.1~0.5)**: 매번 같은 내용, 반복적
- **중간 T (0.7~1.0)**: 자연스럽고 적당히 다양
- **높은 T (1.5~2.0)**: 매번 다른 내용, 때로는 뜬금없음

In [ ]:
# ── Temperature별 출력 비교 ───────────────────────────────────

story_prompt = (
    "Once upon a time in a magical forest,"
)

# ▶ 실험: temperatures 리스트를 원하는 값으로 바꿔보세요
temperatures = [0.1, 0.5, 1.0, 1.5]
RUNS_PER_T = 2  # 각 온도에서 몇 번 반복할지

print(f"프롬프트: '{story_prompt}'")
print("─" * 65)

for T in temperatures:
    print(f"\n  🌡️  Temperature = {T}")
    for run in range(RUNS_PER_T):
        output = generate(
            story_prompt,
            max_new_tokens=25,
            temperature=T,
            top_k=50,
            top_p=0.9,
        )
        print(f"    [{run+1}] {output}")

print()
print("─" * 65)
print("💡 관찰 포인트:")
print("   T=0.1 : 두 번의 출력이 거의 같음 (결정적)")
print("   T=1.5 : 두 번의 출력이 크게 다름 (창의적/무작위적)")

---
## Cell 12 | 심화 실험 — Chain-of-Thought와 역할(Role) 설정

**Chain-of-Thought (CoT)**: "단계별로 생각해라"라는 지시를 프롬프트에 포함시켜  
모델이 중간 추론 과정을 텍스트로 출력하도록 유도하는 기법입니다.

**역할 설정**: 프롬프트 앞에 "You are a ..." 형식으로 모델에 역할을 부여하면  
해당 스타일로 텍스트를 생성하는 경향이 있습니다.

> **주의**: 이 두 기법은 GPT-3.5/4에서 매우 효과적이지만,  
> GPT-2는 지시 따르기 학습이 없어 효과가 제한적입니다.  
> 그럼에도 패턴 추종(In-Context Learning)의 기초 원리는 확인할 수 있습니다.

In [ ]:
# ── ① Chain-of-Thought (CoT) 예시 ───────────────────────────
# Few-shot 예시 안에 "단계별 풀이"를 포함시켜 모델이 같은 방식으로 이어쓰도록 유도

cot_prompt = (
    "Q: John has 5 apples. He gives 2 to Mary. How many does John have?\n"
    "A: John starts with 5. He gives away 2. 5 - 2 = 3. John has 3 apples.\n"
    "\n"
    "Q: Sarah has 8 books. She buys 4 more. How many does she have?\n"
    "A:"  # 모델이 "8 + 4 = 12" 형식의 풀이 과정을 이어쓰기를 기대
)

print("[Chain-of-Thought 프롬프트]")
for line in cot_prompt.strip().split("\n"):
    print(f"  {line}")
cot_result = generate(cot_prompt, max_new_tokens=30, temperature=0.4)
print(f"\n  GPT-2 출력: '{cot_result[:120]}'")

print("\n" + "─" * 60)

# ── ② 역할(Role) 설정 예시 ───────────────────────────────────
# 프롬프트 앞에 "You are a ..." 형식의 역할 지시를 추가

role_prompts = {
    "과학자": (
        "You are a scientist. Explain why the sky is blue:\n"
        "The sky appears blue because"
    ),
    "시인": (
        "You are a poet. Write a short poem about rain:\n"
        "Rain"
    ),
}

print("\n[역할(Role) 설정 예시]")
for role, prompt in role_prompts.items():
    print(f"\n  역할: {role}")
    result = generate(prompt, max_new_tokens=35, temperature=0.8)
    print(f"  출력: '{result[:120]}'")

print("\n─" * 30)
print("💡 GPT-2에서는 역할 지시를 완전히 따르지는 못하지만,")
print("   프롬프트 마지막 단어의 영향을 받아 관련 텍스트를 이어쓰는 것을 볼 수 있습니다.")

---
## Cell 13 | 자유 실험 공간

지금까지 배운 모든 개념을 자유롭게 조합해 보세요.  
아래 변수들을 수정하고 셀을 실행해 결과를 관찰하면 됩니다.

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║                   🔬 자유 실험 공간                      ║
# ║   아래 값들을 원하는 대로 바꾸고 실행해 보세요!          ║
# ╚══════════════════════════════════════════════════════════╝

# ── 1. 프롬프트 수정 ─────────────────────────────────────────
# ▶ Zero-shot / Few-shot 자유롭게 작성
MY_PROMPT = """Classify the following animal:
Dog => Mammal
Eagle => Bird
Salmon => Fish
Python =>"""

# ── 2. 생성 파라미터 조정 ────────────────────────────────────
MY_MAX_NEW_TOKENS = 15   # 생성할 최대 토큰 수
MY_TEMPERATURE    = 0.5  # 낮을수록 결정적 (0.1~2.0)
MY_TOP_K          = 50   # 상위 몇 개 후보만 허용 (1~500)
MY_TOP_P          = 0.9  # 누적 확률 기준 (0.1~1.0)
MY_RUNS           = 3    # 몇 번 생성할지 (다양성 관찰용)

# ── 실행 ──────────────────────────────────────────────────────
print("[내 프롬프트]")
for line in MY_PROMPT.split("\n"):
    print(f"  {line}")
print(f"\n[파라미터] T={MY_TEMPERATURE}, top_k={MY_TOP_K}, top_p={MY_TOP_P}")
print("\n[생성 결과]")
for i in range(MY_RUNS):
    result = generate(
        MY_PROMPT,
        max_new_tokens=MY_MAX_NEW_TOKENS,
        temperature=MY_TEMPERATURE,
        top_k=MY_TOP_K,
        top_p=MY_TOP_P,
    )
    print(f"  [{i+1}] '{result}'")

# ── 힌트 ──────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("💡 다음 실험을 해보세요:")
print("   1. MY_TEMPERATURE = 0.1 vs 2.0 → 출력이 어떻게 달라지나?")
print("   2. MY_TOP_K = 1 → 항상 같은 출력이 나오는가?")
print("   3. Few-shot 예시를 더 추가하면 정확도가 올라가는가?")
print("   4. 한국어 입력을 줬을 때 GPT-2가 어떻게 반응하는가?")

---
## 📚 전체 정리

### 배운 개념 요약

| 개념 | 한 줄 요약 |
|---|---|
| **Logit** | 모델이 각 토큰에 매긴 정규화 이전의 점수 |
| **Softmax** | Logit → 확률 분포로 변환하는 함수 |
| **Temperature** | 확률 분포의 날카로움을 조정 (낮으면 결정적, 높으면 창의적) |
| **Top-k** | 상위 k개 토큰만 후보로 유지 |
| **Top-p** | 누적 확률 p 이내 토큰만 후보로 유지 (동적) |
| **Zero-shot** | 예시 없이 지시만으로 작업 수행 |
| **One/Few-shot** | 1~몇 개 예시로 패턴을 알려줘 작업 수행 |
| **In-Context Learning** | 파라미터 업데이트 없이 맥락만으로 학습하는 현상 |

### GPT-2 → GPT-3 → GPT-4로의 발전

```
GPT-2 (117M~1.5B, 2019)
  → 문장 생성은 잘 하지만, 지시 따르기는 약함
  → Zero-shot 성능 제한적

GPT-3 (175B, 2020)
  → 규모가 커지면서 Few-shot 능력이 급격히 향상
  → 처음으로 Zero-shot도 실용적으로 작동

GPT-3.5 / GPT-4 (2022~2023)
  → RLHF(인간 피드백 강화학습)로 지시 따르기 능력 대폭 강화
  → 역할 설정, Chain-of-Thought 등이 매우 효과적으로 작동
```

### 더 공부하고 싶다면

- [Hugging Face GPT-2 문서](https://huggingface.co/gpt2)
- [OpenAI GPT-3 논문 (Language Models are Few-Shot Learners)](https://arxiv.org/abs/2005.14165)
- [Prompt Engineering Guide](https://www.promptingguide.ai/)